In [2]:
import cv2
from ultralytics import YOLO
from ultralytics.solutions import ObjectCounter

In [3]:
model1 = YOLO("yolov8n.pt")
model2 = YOLO("yolo11n.pt")
model2 = YOLO("yolo12n.pt")

In [4]:
source = "records_combined.mp4"
# source = "0730-1.mp4" 

In [5]:
def get_point_position(point, line_pt1, line_pt2):
    return (line_pt2[0] - line_pt1[0]) * (point[1] - line_pt1[1]) - (line_pt2[1] - line_pt1[1]) * (point[0] - line_pt1[0])

In [6]:
LINE_PT1 = (335, 650)
LINE_PT2 = (390, 0)

In [7]:
cap = cv2.VideoCapture(source)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))      # Ширина в пикселях
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))    # Высота в пикселях
fps = cap.get(cv2.CAP_PROP_FPS)                     # Частота кадров
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))# Общее число кадров

duration_sec = frame_count / fps if fps > 0 else 0
duration_min = duration_sec / 60

print(f"🎬 Разрешение: {width}x{height} px")
print(f"⚡ FPS (частота кадров): {fps:.2f}")
print(f"🎞️ Всего кадров: {frame_count}")
print(f"⏱️ Длительность: {duration_sec:.1f} сек (~{duration_min:.1f} мин)")

cap.release()

🎬 Разрешение: 960x720 px
⚡ FPS (частота кадров): 30.00
🎞️ Всего кадров: 11801
⏱️ Длительность: 393.4 сек (~6.6 мин)


In [8]:
cap = cv2.VideoCapture(source)

if not cap.isOpened():
    print("Ошибка: Не удалось открыть видеофайл.")
    exit()
frame_count = 0
track_history = {}
counted_ids = set()
cross_count = 0 
while True:
    ret, frame = cap.read()
    frame_count += 1
    if frame_count % 5 != 0:
        continue
    # Если ret == False, значит видео закончилось
    if not ret:
        print("Видео завершено.")
        break
    cv2.line(frame, LINE_PT1, LINE_PT2, (255, 0, 0), 2)  
    cv2.putText(frame, "VIRTUAL LINE", (340, 450), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)


    results = model2.track(frame, persist=True, classes=[0], conf=0.5, tracker="botsort.yaml")
    result = results[0]
    
    # 5. Визуализация результатов
    annotated_frame = result.plot()
    
    if result.boxes is not None and result.boxes.id is not None:
        boxes = result.boxes.xyxy.cpu().numpy()
        track_ids = result.boxes.id.int().cpu().numpy()
        for box, track_id in zip(boxes, track_ids):
            x1, y1, x2, y2 = box
            center_x = int((x1 + x2) / 2)
            center_y = int(y2)
            point = (center_x, center_y)
        
            position_val = get_point_position(point, LINE_PT1, LINE_PT2)
            
            if position_val == 0:
                continue
            current_side = position_val > 0
            cv2.putText(annotated_frame, f"{point}, {current_side}", (center_x, center_y), 
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            if track_id not in track_history:
                track_history[track_id] = []
            track_history[track_id].append(current_side)

            if len(track_history[track_id]) > 2:
                track_history[track_id].pop(0)

            if len(track_history[track_id]) == 2:
                prev_side = track_history[track_id][0]
                curr_side = track_history[track_id][1]
            
                if prev_side == False and curr_side == True:
                    if track_id not in counted_ids:
                        cross_count += 1
                    counted_ids.add(track_id)
    cv2.putText(annotated_frame, f"Crossed: {cross_count}", (50, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
    cv2.imshow("YOLOv8 People Tracking", annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    # 6. Отображение кадра на экране
    cv2.imshow("YOLOv8 People Tracking", annotated_frame)
    
    # Выход по нажатию клавиши 'q' (задержка 1 мс между кадрами)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
        
# 7. Очистка ресурсов
cap.release()
cv2.destroyAllWindows()


0: 480x640 (no detections), 612.0ms
Speed: 9.7ms preprocess, 612.0ms inference, 13.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 421.1ms
Speed: 16.1ms preprocess, 421.1ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 218.0ms
Speed: 5.8ms preprocess, 218.0ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 205.7ms
Speed: 28.3ms preprocess, 205.7ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 158.5ms
Speed: 5.4ms preprocess, 158.5ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 138.7ms
Speed: 4.8ms preprocess, 138.7ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 163.2ms
Speed: 4.6ms preprocess, 163.2ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 160.4ms
Speed: 4.5ms pr

### Проверка правильности определения знака точки на линии

In [9]:
video_path = '0730-1.mp4'  # Укажите путь к вашему видео
image_path = 'screenshot.jpg' # Как будет называться сохраненный файл

cap = cv2.VideoCapture(video_path)

if cap.isOpened():
    ret, frame = cap.read() # Читаем первый кадр
    
    if ret:
        # Сохраняем массив (кадр) как JPG файл
        cv2.imwrite(image_path, frame)
        print(f"Скриншот успешно сохранен как {image_path}")
    else:
        print("Не удалось прочитать кадр.")
else:
    print("Не удалось открыть видео.")

cap.release()

Скриншот успешно сохранен как screenshot.jpg


In [10]:
import numpy as np
def click_event(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:  # Если нажата левая кнопка мыши
        point = (x, y)
        
        # Вычисляем математическое значение для точки клика
        val = get_point_position(point, LINE_PT1, LINE_PT2)
        
        # Определяем знак и цвет для визуализации
        if val > 0:
            text = "POSITIVE (+)"
            color = (0, 255, 0)  # Зеленый цвет для плюса
        elif val < 0:
            text = "NEGATIVE (-)"
            color = (0, 0, 255)  # Красный цвет для минуса
        else:
            text = "ON THE LINE (0)"
            color = (255, 255, 0) # Голубой для точного попадания на линию
            
        # Выводим в консоль точные значения
        print(f"Клик по координатам (x={x}, y={y}). Значение функции: {val} -> {text}")
        
        # Рисуем точку клика
        cv2.circle(img, point, 5, color, -1)
        # Подписываем рядом с точкой результат (+ или -)
        cv2.putText(img, text, (x + 10, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        
        # Обновляем картинку на экране
        cv2.imshow("Line Test", img)

# 1. Загрузка картинки
# Замените 'screenshot.jpg' на имя вашего файла.
image_path = 'screenshot.jpg' 
img = cv2.imread(image_path)

# Если скриншот не найден в папке, создаем просто пустой черный фон нужного размера
if img is None:
    print(f"Не удалось загрузить {image_path}. Создан черный фон.")
    img = np.zeros((700, 800, 3), dtype=np.uint8)

# 2. Рисуем саму линию, чтобы понимать, где она находится
cv2.line(img, LINE_PT1, LINE_PT2, (255, 0, 0), 2)
cv2.circle(img, LINE_PT1, 5, (255, 0, 0), -1)
cv2.circle(img, LINE_PT2, 5, (255, 0, 0), -1)
cv2.putText(img, "LINE", (LINE_PT1[0] - 40, LINE_PT1[1] + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

# 3. Настройка окна и "прослушивателя" мыши
cv2.namedWindow("Line Test")
cv2.setMouseCallback("Line Test", click_event)

print("Окно открыто. Кликайте ЛЕВОЙ кнопкой мыши по экрану.")
print("Для выхода нажмите английскую клавишу 'q'.")

# 4. Основной цикл отображения
cv2.imshow("Line Test", img)
while True:
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

Окно открыто. Кликайте ЛЕВОЙ кнопкой мыши по экрану.
Для выхода нажмите английскую клавишу 'q'.
Клик по координатам (x=314, y=201). Значение функции: -38345 -> NEGATIVE (-)
Клик по координатам (x=453, y=243). Значение функции: 54315 -> POSITIVE (+)


## Обрезка видео

In [11]:
# Укажите путь к вашему исходному видео
video_path = source 

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Ошибка: Не удалось открыть видеофайл.")
    exit()

# Читаем первый кадр видео
ret, frame = cap.read()
cap.release()

if ret:
    print("Окно открыто. Выделите нужную область мышкой.")
    print("Для подтверждения нажмите ENTER или ПРОБЕЛ. Для отмены нажмите 'c'.")
    
    # Открываем интерактивное окно для выбора зоны (Region Of Interest)
    # showCrosshair=True показывает перекрестие для удобства
    roi = cv2.selectROI("Select Crop Region", frame, showCrosshair=True, fromCenter=False)
    cv2.destroyAllWindows()
    
    # roi возвращает кортеж: (x, y, ширина, высота)
    x, y, w, h = roi
    
    if w == 0 or h == 0:
        print("Вы ничего не выделили.")
    else:
        print("\n=== ВАШИ КООРДИНАТЫ ДЛЯ ШАГА 2 ===")
        print(f"X (отступ слева): {x}")
        print(f"Y (отступ сверху): {y}")
        print(f"W (ширина): {w}")
        print(f"H (высота): {h}")
        print("Скопируйте эти значения во второй скрипт!")
else:
    print("Не удалось прочитать кадр из видео.")

Окно открыто. Выделите нужную область мышкой.
Для подтверждения нажмите ENTER или ПРОБЕЛ. Для отмены нажмите 'c'.

=== ВАШИ КООРДИНАТЫ ДЛЯ ШАГА 2 ===
X (отступ слева): 198
Y (отступ сверху): 9
W (ширина): 278
H (высота): 436
Скопируйте эти значения во второй скрипт!


In [12]:
import numpy as np

def crop_frame(frame: np.ndarray, x: int, y: int, w: int, h: int) -> np.ndarray:
    """
    Вырезает прямоугольную область из кадра.
    
    :param frame: Исходный кадр (NumPy array)
    :param x: Начальная координата X (левый край)
    :param y: Начальная координата Y (верхний край)
    :param w: Ширина вырезаемой области
    :param h: Высота вырезаемой области
    :return: Обрезанный фрагмент кадра (ROI)
    """
    # Защита от выхода координат за границы изображения
    img_h, img_w = frame.shape[:2]
    x1 = max(0, x)
    y1 = max(0, y)
    x2 = min(img_w, x + w)
    y2 = min(img_h, y + h)
    
    return frame[y1:y2, x1:x2]

In [ ]:
# 2. Определение зоны подсчета трафика (ROI) на кадре 960x720
# Например: входная группа по центру кадра
ROI_X, ROI_Y, ROI_W, ROI_H = 198, 9, 278, 436

# 3. Виртуальная линия подсчета (внутри координат ROI)
LINE_Y = 250
counted_ids = set()

# Источник: RTSP-поток или видеофайл
cap = cv2.VideoCapture("video_960x720.mp4")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Вырезаем область для нейросети
    roi_crop = crop_frame(frame, ROI_X, ROI_Y, ROI_W, ROI_H)

    # Передаем в YOLO только обрезанную часть кадра
    results = model.track(roi_crop, persist=True, classes=[0], tracker="bytetrack.yaml", verbose=False)

    # Отрисовываем границы зоны анализа на основном кадре
    cv2.rectangle(frame, (ROI_X, ROI_Y), (ROI_X + ROI_W, ROI_Y + ROI_H), (255, 255, 0), 2)
    cv2.putText(frame, "Зона детекции (ROI)", (ROI_X, ROI_Y - 8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        track_ids = results[0].boxes.id.int().cpu().tolist()

        for box, track_id in zip(boxes, track_ids):
            bx1, by1, bx2, by2 = map(int, box)
            center_y_roi = (by1 + by2) // 2

            # Логика фиксации прохода внутри ROI
            if center_y_roi > LINE_Y and track_id not in counted_ids:
                counted_ids.add(track_id)

            # Пересчитываем координаты рамки на полный исходный кадр
            global_x1 = bx1 + ROI_X
            global_y1 = by1 + ROI_Y
            global_x2 = bx2 + ROI_X
            global_y2 = by2 + ROI_Y

            # Рисуем рамку человека на основном кадре
            cv2.rectangle(frame, (global_x1, global_y1), (global_x2, global_y2), (0, 255, 0), 2)
            cv2.putText(frame, f"ID: {track_id}", (global_x1, global_y1 - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Отрисовка счетчика посетителей
    cv2.putText(frame, f"Посетителей: {len(counted_ids)}", (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 0, 255), 3)

    cv2.imshow("TRASSIR + YOLO ROI Tracking", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()